# Introduction
The dataset is from the Kaggle platform. The dataset discusses the customer complaints and teams involved in resolving this, with data size of 85,907 "customer service interactions" .As a part of building my portfolio, I used SQL and python to explore what kind of insights can be extracted from customer service data.

here we try to find the followings from dataset
1) Which category has most complaints
2) which shift is better morning or evening ?
3) which channel gets most contact?
4) Which category has the lowest average CSAT score?
5) What percentage of contacts come from each channel?
6) Which manager has the best performing team? 

In [2]:
import pandas as pd
import sqlite3

In [3]:
df = pd.read_csv('Customer_support_data.csv')
conn = sqlite3.connect(':memory:')
df.to_sql('Customer_Data',conn)
df_from_db = pd.read_sql('SELECT * FROM Customer_Data', conn)
print(df_from_db.head())

   index                             Unique id channel_name         category  \
0      0  7e9ae164-6a8b-4521-a2d4-58f7c9fff13f      Outcall  Product Queries   
1      1  b07ec1b0-f376-43b6-86df-ec03da3b2e16      Outcall  Product Queries   
2      2  200814dd-27c7-4149-ba2b-bd3af3092880      Inbound    Order Related   
3      3  eb0d3e53-c1ca-42d3-8486-e42c8d622135      Inbound          Returns   
4      4  ba903143-1e54-406c-b969-46c52f92e5df      Inbound     Cancellation   

                   Sub-category Customer Remarks  \
0                Life Insurance              NaN   
1  Product Specific Information              NaN   
2             Installation/demo              NaN   
3        Reverse Pickup Enquiry              NaN   
4                    Not Needed              NaN   

                               Order_id order_date_time Issue_reported at  \
0  c27c9bb4-fa36-4140-9f1f-21009254ffdb             NaN  01/08/2023 11:13   
1  d406b0c7-ce17-4654-b9de-f08d421254bd             

In [61]:
df_from_db = pd.read_sql("""
                         Select category , COUNT(*) AS count_category
                         From Customer_Data
                         Group by category
                         Order by count_category DESC;
                         """, conn)
print(df_from_db)

              category  count_category
0              Returns           44097
1        Order Related           23215
2       Refund Related            4550
3      Product Queries            3692
4    Shopzilla Related            2792
5     Payments related            2327
6             Feedback            2294
7         Cancellation            2212
8    Offers & Cashback             480
9               Others              99
10         App/website              84
11  Onboarding related              65


In [5]:
df_from_db = pd.read_sql("""
                         Select "Agent Shift",Avg("CSAT Score") AS AVG_CSAT
                         From Customer_Data
                         Group by "Agent Shift"
                         order by AVG_CSAT DESC
                         Limit 5;
                         """, conn)
print(df_from_db)

  Agent Shift  AVG_CSAT
0       Split  4.427357
1   Afternoon  4.292295
2       Night  4.288754
3     Evening  4.276361
4     Morning  4.189495


In [6]:
df_from_db = pd.read_sql("""
                         Select channel_name , COUNT(channel_name) AS count_Channel
                         From Customer_Data
                         Group by channel_name
                         Order by count_Channel DESC;
                         """, conn)
print(df_from_db)

  channel_name  count_Channel
0      Inbound          68142
1      Outcall          14742
2        Email           3023


In [7]:
df_from_db = pd.read_sql("""
                         Select category ,Avg("CSAT Score") AS AVG_CSAT
                         From Customer_Data
                         Group by category
                         order by AVG_CSAT ASC
                         Limit 5;
                         """, conn)
print(df_from_db)

          category  AVG_CSAT
0           Others  3.434343
1     Cancellation  3.990506
2  Product Queries  4.039816
3    Order Related  4.096059
4         Feedback  4.158675


In [39]:
df_from_db = pd.read_sql("""
                         Select channel_name  ,Count(channel_name)*100.00/(Select Count(*) From Customer_Data) as percentage
                         From Customer_Data
                         Group by channel_name
                         Limit 5;
                         """, conn)
print(df_from_db)

  channel_name  percentage
0        Email    3.518922
1      Inbound   79.320661
2      Outcall   17.160418


In [55]:
df_from_db = pd.read_sql("""
                         Select Manager,Agent_name ,Avg("CSAT Score") AS AVG_CSAT
                         From Customer_Data
                         Where Manager= 'Emily Chen'
                         Group by Agent_name
                         order by AVG_CSAT DESC
                         Limit 10;
                         """, conn)
print(df_from_db)

      Manager         Agent_name  AVG_CSAT
0  Emily Chen    Pamela Robinson  4.956522
1  Emily Chen      Taylor Nelson  4.888889
2  Emily Chen        Nancy Singh  4.868421
3  Emily Chen       Morgan Smith  4.866667
4  Emily Chen    Emily Hernandez  4.843750
5  Emily Chen       Linda Murray  4.837500
6  Emily Chen  Kelsey Richardson  4.830189
7  Emily Chen          Penny Lam  4.821053
8  Emily Chen  Dr. Heather Lewis  4.813333
9  Emily Chen       Robert Lewis  4.801325


## Insights 
- Returns dominate complaints at 44,097 — more than all other categories combined
- Inbound calls are 79.3% of all contacts — email is barely used at 3.5%
- Split shift agents outperform morning shift agents on CSAT
- Emily Chen's team is the best performing — led by Pamela Robinson at 4.96/5
- "Others" category has the lowest CSAT at 3.43 — unclassified issues frustrate customers most
   